# Empirical Analysis: Parameter-Efficient Fine-Tuning (PEFT) Benchmark

This research notebook presents a comprehensive analysis of Parameter-Efficient Fine-Tuning (PEFT) methods (**LoRA**, **AdaLoRA**, **Prefix Tuning**, **IA³**) compared against **Full Fine-Tuning** across **BERT-base** and **DistilBERT** on GLUE classification benchmarks (**SST-2**, **MRPC**, **RTE**).

## Notebook Structure
1. **Loading Results & Manifest Validation**
2. **Official GLUE Benchmark Metrics**
3. **Training Cost & Analytical FLOPs**
4. **Memory Cost & NVML Hardware Utilization**
5. **Parameter Efficiency & Checkpoint Footprint**
6. **Inference Latency & Throughput**
7. **Stability Across Seeds & Statistical Significance**
8. **Per-Backbone Pareto Frontier Analysis**
9. **Failure Analysis**
10. **Limitations & Conclusions**

## 1. Loading Results & Manifest Validation

In [ ]:
import os
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set Matplotlib publication style
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['grid.color'] = '#cccccc'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.alpha'] = 0.5

RESULTS_DIR = '../results'
if not os.path.exists(RESULTS_DIR):
    RESULTS_DIR = 'results'

manifest_path = os.path.join(RESULTS_DIR, 'manifest.json')
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    print(f'Loaded manifest index containing {len(manifest)} experiment runs.')
else:
    print('No manifest.json found. Please run scripts/run_full_matrix.py to generate results.')


## 2. Official GLUE Benchmark Metrics

In [ ]:
stats_path = os.path.join(RESULTS_DIR, 'statistical_tests.json')
if os.path.exists(stats_path):
    with open(stats_path, 'r') as f:
        stats_data = json.load(f)
    
    table_rows = []
    for pair_key, methods in stats_data.get('comparisons', {}).items():
        model, dataset = pair_key.split('__')
        for method, info in methods.items():
            table_rows.append({
                'Model': model,
                'Dataset': dataset.upper(),
                'Method': method,
                'Mean Acc': f"{info['mean_accuracy']:.4f}",
                'Std Dev': f"{info['std_accuracy']:.4f}",
                '95% CI': f"±{info['ci95_accuracy']:.4f}"
            })
    
    df_summary = pd.DataFrame(table_rows)
    print(df_summary.to_string(index=False))


## 3. Training Cost & Analytical FLOPs

In [ ]:
# Plot Analytical FLOPs comparison using Matplotlib
def plot_flops_comparison(results_dir):
    metrics_files = glob.glob(os.path.join(results_dir, '**', 'metrics.json'), recursive=True)
    data = []
    for mf in metrics_files:
        with open(mf, 'r') as f:
            d = json.load(f)
            eff = d.get('efficiency_metrics', {})
            parts = mf.split(os.sep)
            # Extract model, dataset, method
            method = parts[-3]
            dataset = parts[-4]
            model = parts[-5]
            data.append({
                'model': model,
                'dataset': dataset,
                'method': method,
                'flops': eff.get('approximate_analytical_flops_estimate', 0)
            })
    df = pd.DataFrame(data)
    if not df.empty:
        pivot = df.groupby(['model', 'method'])['flops'].mean().unstack(level=0)
        fig, ax = plt.subplots(figsize=(8, 4))
        pivot.plot(kind='bar', ax=ax, colormap='Accent', edgecolor='black')
        ax.set_ylabel('Approximate Analytical FLOPs')
        ax.set_title('Training FLOPs by Method and Backbone')
        ax.grid(True, axis='y')
        plt.tight_layout()
        plt.show()

plot_flops_comparison(RESULTS_DIR)


## 4. Memory Cost & NVML Hardware Utilization

In [ ]:
# Plot Peak VRAM Memory consumption
def plot_vram_usage(results_dir):
    metrics_files = glob.glob(os.path.join(results_dir, '**', 'metrics.json'), recursive=True)
    data = []
    for mf in metrics_files:
        with open(mf, 'r') as f:
            d = json.load(f)
            eff = d.get('efficiency_metrics', {})
            parts = mf.split(os.sep)
            data.append({
                'model': parts[-5],
                'method': parts[-3],
                'peak_vram_mb': eff.get('peak_vram_mb', 0)
            })
    df = pd.DataFrame(data)
    if not df.empty:
        pivot = df.groupby(['model', 'method'])['peak_vram_mb'].mean().unstack(level=0)
        fig, ax = plt.subplots(figsize=(8, 4))
        pivot.plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e'], edgecolor='black')
        ax.set_ylabel('Peak VRAM (MB)')
        ax.set_title('Peak VRAM Footprint by Method')
        ax.grid(True, axis='y')
        plt.tight_layout()
        plt.show()

plot_vram_usage(RESULTS_DIR)


## 5. Parameter Efficiency & Checkpoint Footprint

In [ ]:
# Display parameter percentage table
metrics_files = glob.glob(os.path.join(RESULTS_DIR, '**', 'metrics.json'), recursive=True)
param_data = []
for mf in metrics_files:
    with open(mf, 'r') as f:
        d = json.load(f)
        p = d.get('parameter_metrics', {})
        parts = mf.split(os.sep)
        param_data.append({
            'Model': parts[-5],
            'Method': parts[-3],
            'Trainable Params': p.get('trainable_parameters', 0),
            'Total Params': p.get('total_parameters', 0),
            '% Trainable': f"{p.get('pct_trainable', 0):.3f}%",
            'Checkpoint (MB)': f"{p.get('checkpoint_size_mb', 0):.2f}"
        })
if param_data:
    df_params = pd.DataFrame(param_data).drop_duplicates(subset=['Model', 'Method'])
    print(df_params.to_string(index=False))


## 6. Inference Latency & Throughput

In [ ]:
# Plot Inference Latency
def plot_latency(results_dir):
    metrics_files = glob.glob(os.path.join(results_dir, '**', 'metrics.json'), recursive=True)
    data = []
    for mf in metrics_files:
        with open(mf, 'r') as f:
            d = json.load(f)
            eff = d.get('efficiency_metrics', {})
            parts = mf.split(os.sep)
            data.append({
                'model': parts[-5],
                'method': parts[-3],
                'latency_ms': eff.get('inference_latency_ms_per_sample', 0)
            })
    df = pd.DataFrame(data)
    if not df.empty:
        pivot = df.groupby(['model', 'method'])['latency_ms'].mean().unstack(level=0)
        fig, ax = plt.subplots(figsize=(8, 4))
        pivot.plot(kind='bar', ax=ax, color=['#2ca02c', '#d62728'], edgecolor='black')
        ax.set_ylabel('Inference Latency (ms / sample)')
        ax.set_title('Inference Latency per Sample by Method')
        ax.grid(True, axis='y')
        plt.tight_layout()
        plt.show()

plot_latency(RESULTS_DIR)


## 7. Stability Across Seeds & Statistical Significance

In [ ]:
# Display exploratory statistical tests vs Full Fine-Tuning
if os.path.exists(stats_path):
    with open(stats_path, 'r') as f:
        stats_data = json.load(f)
    print("Methodology Note:", stats_data.get('methodology_note'))
    print("\nExploratory Comparisons vs Full Fine-Tuning:")
    for pair, methods in stats_data.get('comparisons', {}).items():
        print(f"\n--- {pair} ---")
        for m, info in methods.items():
            vs = info.get('vs_full_fine_tuning')
            if vs:
                print(f"Method {m:<8} | Mean Diff: {vs['mean_difference']:+.4f} | Cohen's d: {vs['cohens_d']:+.2f} | t-test p: {vs['exploratory_paired_ttest_pvalue']}")


## 8. Per-Backbone Pareto Frontier Analysis

In [ ]:
pareto_path = os.path.join(RESULTS_DIR, 'pareto.json')
if os.path.exists(pareto_path):
    with open(pareto_path, 'r') as f:
        pareto_data = json.load(f)
    
    for group_key, items in pareto_data.get('backbones', {}).items():
        print(f"\n--- Pareto Frontier for {group_key} ---")
        df_p = pd.DataFrame(items)
        print(df_p[['method', 'mean_accuracy', 'mean_peak_vram_mb', 'mean_training_time_seconds', 'pareto_optimal']].to_string(index=False))


## 9. Failure Analysis

In [ ]:
# Load sample misclassified predictions
misc_files = glob.glob(os.path.join(RESULTS_DIR, '**', 'misclassified.csv'), recursive=True)
if misc_files:
    df_misc = pd.read_csv(misc_files[0])
    print(f"Sample Misclassified Predictions from {misc_files[0]}:")
    print(df_misc.head(10).to_string(index=False))
else:
    print("No misclassified.csv files found.")


## 10. Limitations & Conclusions

### Framework Limitations
1. **Architecture Boundary**: Evaluated strictly on encoder-only transformer backbones (`bert-base-uncased`, `distilbert-base-uncased`).
2. **Task Domain**: Benchmark restricted to English text classification tasks from GLUE (SST-2, MRPC, RTE).
3. **Hardware Scope**: Empirical measurements gathered on single-GPU hardware (NVIDIA RTX 3050 Laptop GPU, 6 GB VRAM).
4. **Hyperparameter Policy**: Uniformly controlled learning rate and training schedules applied across all methods to isolate adaptation efficiency.
5. **FLOPs Estimation**: FLOPs reported are analytical estimates ($6 \times N_{\text{params}} \times N_{\text{tokens}}$) rather than hardware counter measurements.

### Summary Conclusions
- Parameter-Efficient Fine-Tuning methods achieve competitive predictive accuracy relative to Full Fine-Tuning while updating less than 1% of total parameters.
- Checkpoint storage is reduced by orders of magnitude (from ~400 MB to ~1 MB), rendering PEFT methods highly advantageous for multi-task deployments.